In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error


In [2]:
# Load your exact file
df = pd.read_csv("car data.csv")

# Print verification information
print("Confirmed Columns:", df.columns.tolist())
print("\nFirst 5 rows of data:")
print(df.head())


Confirmed Columns: ['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner']

First 5 rows of data:
  Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Selling_type Transmission  Owner  
0       Dealer       Manual      0  
1       Dealer       Manual      0  
2       Dealer       Manual      0  
3       Dealer       Manual      0  
4       Dealer       Manual      0  


In [3]:
# Calculate the vehicle age based on the current year (2026)
df['Current_Year'] = 2026
df['Car_Age'] = df['Current_Year'] - df['Year']

# Drop the columns we no longer need to feed directly into the model
df = df.drop(columns=['Year', 'Current_Year', 'Car_Name'])

print("Engineered Data View:")
print(df[['Car_Age', 'Selling_Price']].head())


Engineered Data View:
   Car_Age  Selling_Price
0       12           3.35
1       13           4.75
2        9           7.25
3       15           2.85
4       12           4.60


In [4]:
X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

print("Input features columns:", X.columns.tolist())


Input features columns: ['Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Car_Age']


In [5]:
# Separate columns into categoricals and numericals
categorical_cols = ['Fuel_Type', 'Selling_type', 'Transmission']
numerical_cols = ['Present_Price', 'Driven_kms', 'Owner', 'Car_Age']

# Initialize the transformation pipeline rules
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)


In [6]:
# Split the dataset into 80% for training and 20% for testing validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Matrix Size: {X_train.shape}")
print(f"Testing Matrix Size: {X_test.shape}")


Training Matrix Size: (240, 7)
Testing Matrix Size: (61, 7)


In [7]:
# Construct an end-to-end processing and modeling framework
car_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit/Train the model
car_model_pipeline.fit(X_train, y_train)
print("Car valuation regression training complete!")


Car valuation regression training complete!


In [8]:
# Generate predictions across test parameters
y_pred = car_model_pipeline.predict(X_test)

# Print accuracy evaluations
print(f"Model R2 Score (Accuracy): {r2_score(y_test, y_pred) * 100:.2f}%")
print(f"Mean Absolute Error (Average price deviation): {mean_absolute_error(y_test, y_pred):.2f} Lakhs")


Model R2 Score (Accuracy): 95.86%
Mean Absolute Error (Average price deviation): 0.64 Lakhs


In [9]:
# Define custom features mapping exact column layouts
custom_car_spec = pd.DataFrame([{
    'Present_Price': 9.54,
    'Driven_kms': 43000,
    'Fuel_Type': 'Diesel',
    'Selling_type': 'Dealer',
    'Transmission': 'Manual',
    'Owner': 0,
    'Car_Age': 7
}], columns=X.columns)

# Run valuation pipeline
predicted_valuation = car_model_pipeline.predict(custom_car_spec)
print(f"Estimated Market Selling Price: {predicted_valuation[0]:.2f} Lakhs")


Estimated Market Selling Price: 7.97 Lakhs
